In [1]:
import json
from pathlib import Path

archivo_enero = Path("../data/raw/2025/releases_2025_enero_subasta_inversa_electronica.json")

with open(archivo_enero, "r", encoding="utf-8") as archivo:
    datos_enero = json.load(archivo)

type(datos_enero)

list

In [2]:
len(datos_enero)

91

#### Identificar el tipo de cada registro

Se analiza el primer elemento de la lista para verificar si cada registro está almacenado como un diccionario de Python.

In [7]:
type(datos_enero[0])

dict

#### Identificar los campos disponibles

Se muestran las claves del primer registro para conocer qué variables contiene la estructura del archivo JSON.

In [6]:
datos_enero[0].keys()

dict_keys(['uri', 'license', 'version', 'releases', 'publisher', 'extensions', 'publishedDate', 'publicationPolicy'])

#### Explorar la lista de releases

Se accede a la clave `releases`, donde se encuentran los registros individuales de contratación pública contenidos en el paquete.

In [5]:
type(datos_enero[0]["releases"])

list

In [8]:
len(datos_enero[0]["releases"])

1

#### Identificar los campos del primer release

Se accede al primer registro de contratación para conocer las variables disponibles dentro de cada procedimiento publicado.

In [9]:
datos_enero[0]["releases"][0].keys()

dict_keys(['id', 'tag', 'date', 'ocid', 'buyer', 'tender', 'parties', 'language', 'planning', 'initiationType'])

#### Explorar la entidad compradora

Se revisa la estructura del campo `buyer` para identificar la entidad contratante y los datos disponibles para construir los nodos de compradores.

In [10]:
datos_enero[0]["releases"][0]["buyer"]

{'id': 'EC-RUC-1560001400001-77741', 'name': 'Municipio El Chaco'}

#### Explorar la información del procedimiento

Se revisa la estructura del campo `tender` para identificar datos como el código del proceso, estado, descripción, monto y método de contratación.

In [11]:
datos_enero[0]["releases"][0]["tender"].keys()

dict_keys(['id', 'lots', 'title', 'status', 'enquiries', 'tenderers', 'awardPeriod', 'description', 'hasEnquiries', 'tenderPeriod', 'awardCriteria', 'enquiryPeriod', 'procuringEntity', 'numberOfTenderers', 'procurementMethod', 'mainProcurementCategory', 'procurementMethodDetails'])

#### Revisar los valores principales del procedimiento

Se consultan los campos más relevantes del primer procedimiento para conocer cómo están representados el identificador, título, estado, método de contratación y número de oferentes.

In [12]:
tender = datos_enero[0]["releases"][0]["tender"]

{
    "id": tender.get("id"),
    "title": tender.get("title"),
    "status": tender.get("status"),
    "procurementMethod": tender.get("procurementMethod"),
    "procurementMethodDetails": tender.get("procurementMethodDetails"),
    "numberOfTenderers": tender.get("numberOfTenderers")
}

{'id': 'SIE-CHACO-2025-009-77741',
 'title': 'SIE-CHACO-2025-009-77741',
 'status': 'active',
 'procurementMethod': 'open',
 'procurementMethodDetails': 'Subasta Inversa Electrónica',
 'numberOfTenderers': 4}

#### Explorar los oferentes del procedimiento

Se revisa el campo `tenderers` para identificar a los proveedores que participaron en el procedimiento y los datos disponibles para construir los nodos de oferentes.

In [13]:
type(tender["tenderers"]), len(tender["tenderers"])

(list, 4)

#### Identificar los campos de los oferentes

Se analiza el primer oferente para conocer qué datos están disponibles sobre los proveedores participantes.

In [14]:
tender["tenderers"][0].keys()

dict_keys(['id', 'name'])

#### Visualizar los oferentes participantes

Se muestran los identificadores y nombres de todos los oferentes del primer procedimiento para verificar cómo se representan los proveedores participantes.

In [15]:
tender["tenderers"]

[{'id': 'EC-RUC-1600412769001-311697', 'name': 'ALVEAR AREVALO DIEGO ANDRES'},
 {'id': 'EC-RUC-1600537151001-440152', 'name': 'SOLIS ROBALINO MELANI MAYTE'},
 {'id': 'EC-RUC-1792131480001-80578', 'name': 'RAMFORTRADE CIA. LTDA.'},
 {'id': 'EC-RUC-1792734789001-822600',
  'name': 'COMERCIALIZADORA TRACTOR-ZONE CIA LTDA'}]

#### Explorar las partes relacionadas

Se revisa el campo `parties` para identificar las organizaciones involucradas en el procedimiento y los roles que desempeñan.

In [16]:
release = datos_enero[0]["releases"][0]

type(release["parties"]), len(release["parties"])

(list, 5)

#### Identificar la estructura de las partes

Se analiza la primera organización de la lista `parties` para conocer sus identificadores, nombre y roles dentro del procedimiento.

In [17]:
release["parties"][0].keys()

dict_keys(['id', 'name', 'roles', 'address', 'identifier', 'contactPoint'])

#### Identificar los roles de las organizaciones

Se muestran el identificador, nombre y roles de cada organización para distinguir a la entidad compradora de los proveedores participantes.

In [18]:
[
    {
        "id": parte.get("id"),
        "name": parte.get("name"),
        "roles": parte.get("roles")
    }
    for parte in release["parties"]
]

[{'id': 'EC-RUC-1560001400001-77741',
  'name': 'Municipio El Chaco',
  'roles': ['buyer', 'procuringEntity']},
 {'id': 'EC-RUC-1600412769001-311697',
  'name': 'ALVEAR AREVALO DIEGO ANDRES',
  'roles': ['tenderer']},
 {'id': 'EC-RUC-1600537151001-440152',
  'name': 'SOLIS ROBALINO MELANI MAYTE',
  'roles': ['tenderer']},
 {'id': 'EC-RUC-1792131480001-80578',
  'name': 'RAMFORTRADE CIA. LTDA.',
  'roles': ['tenderer']},
 {'id': 'EC-RUC-1792734789001-822600',
  'name': 'COMERCIALIZADORA TRACTOR-ZONE CIA LTDA',
  'roles': ['tenderer']}]

#### Verificar la existencia de información de adjudicación

Se revisan las claves de todos los releases para determinar si existe el campo `awards`, donde normalmente se identifica al proveedor adjudicado.

In [19]:
claves_releases = set()

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        claves_releases.update(release_item.keys())

sorted(claves_releases)

['auctions',
 'awards',
 'buyer',
 'contracts',
 'date',
 'id',
 'initiationType',
 'language',
 'ocid',
 'parties',
 'planning',
 'tag',
 'tender']

#### Contar los releases con información de adjudicación

Se cuenta cuántos releases de enero contienen el campo `awards`, con el fin de identificar los procedimientos que ya incluyen información sobre proveedores adjudicados.

In [20]:
releases_con_awards = []

for paquete in datos_enero:
    for release_item in paquete.get("releases", []):
        if release_item.get("awards"):
            releases_con_awards.append(release_item)

len(releases_con_awards)

78

#### Explorar la estructura de las adjudicaciones

Se analiza el primer release que contiene información de adjudicación para identificar los campos disponibles sobre el proveedor ganador, el monto y el estado de la adjudicación.

In [21]:
releases_con_awards[0]["awards"]

[{'id': '2428016-SIE-HEP-2024-00115',
  'date': '2025-02-28T20:00:35-05:00',
  'items': [{'id': '4263344-DS-SO',
    'unit': {'id': '436', 'name': 'Unidad', 'scheme': 'SERCOP'},
    'quantity': 6600,
    'description': 'INSUMOS DE USO GENERAL',
    'classification': {'id': '352901091',
     'scheme': 'CPC',
     'description': 'INSUMOS DE USO GENERAL'},
    'additionalClassifications': [{'id': '35290.10.9',
      'uri': 'https://www.compraspublicas.gob.ec/ProcesoContratacion/compras/exe/verComProductos_exe.php?tipo=buscar&idProducto=35290.10.9',
      'scheme': 'CPC',
      'description': 'INSUMOS MEDICOS HOSPITALARIOS'}]}],
  'value': {'amount': 243500, 'currency': 'USD'},
  'suppliers': [{'id': 'EC-RUC-1312665100001-1217500',
    'name': 'ZAMBRANO VELEZ JEAN PIERRE'}],
  'description': 'Se resuelve ADJUDICAR el proceso de Subasta Inversa Electrónica Nro. SIE-HEP-2024-00115, cuyo objeto es la "ADQUISICIÓN DE DISPOSITIVOS MÉDICOS - EQUIPO DE PROTECCIÓN PERSONAL PARA LA ATENCIÓN OPORTUN

#### Extraer los datos principales de la adjudicación

Se seleccionan el identificador de la adjudicación, la fecha, el monto, la moneda y el proveedor adjudicado para comprobar cómo se estructurarán las relaciones de adjudicación en el modelo de red.

In [22]:
award = releases_con_awards[0]["awards"][0]

{
    "award_id": award.get("id"),
    "fecha": award.get("date"),
    "monto": award.get("value", {}).get("amount"),
    "moneda": award.get("value", {}).get("currency"),
    "proveedores": award.get("suppliers", [])
}

{'award_id': '2428016-SIE-HEP-2024-00115',
 'fecha': '2025-02-28T20:00:35-05:00',
 'monto': 243500,
 'moneda': 'USD',
 'proveedores': [{'id': 'EC-RUC-1312665100001-1217500',
   'name': 'ZAMBRANO VELEZ JEAN PIERRE'}]}

#### Contar procedimientos únicos mediante el OCID

Se recopilan los identificadores `ocid` de todos los releases para determinar cuántos procedimientos de contratación únicos existen y verificar si un mismo procedimiento aparece en varias etapas.

In [23]:
todos_los_releases = [
    release_item
    for paquete in datos_enero
    for release_item in paquete.get("releases", [])
]

ocids = [
    release_item.get("ocid")
    for release_item in todos_los_releases
    if release_item.get("ocid")
]

{
    "total_releases": len(todos_los_releases),
    "procedimientos_unicos": len(set(ocids)),
    "releases_repetidos": len(ocids) - len(set(ocids))
}

{'total_releases': 91, 'procedimientos_unicos': 91, 'releases_repetidos': 0}

#### Contar proveedores adjudicados por procedimiento

Se analiza cuántos proveedores aparecen en cada adjudicación para identificar si los procedimientos tienen un único ganador o varios proveedores adjudicados.

In [24]:
cantidad_proveedores_por_award = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        cantidad_proveedores_por_award.append(
            len(award_item.get("suppliers", []))
        )

{
    "total_awards": len(cantidad_proveedores_por_award),
    "minimo_proveedores": min(cantidad_proveedores_por_award),
    "maximo_proveedores": max(cantidad_proveedores_por_award),
    "distribucion": {
        cantidad: cantidad_proveedores_por_award.count(cantidad)
        for cantidad in sorted(set(cantidad_proveedores_por_award))
    }
}

{'total_awards': 78,
 'minimo_proveedores': 1,
 'maximo_proveedores': 1,
 'distribucion': {1: 78}}

#### Analizar los montos adjudicados

Se extraen los montos de todas las adjudicaciones para conocer el valor mínimo, máximo, promedio y total adjudicado durante el período analizado.

In [25]:
montos_adjudicados = []

for release_item in releases_con_awards:
    for award_item in release_item.get("awards", []):
        monto = award_item.get("value", {}).get("amount")

        if monto is not None:
            montos_adjudicados.append(monto)

{
    "cantidad_montos": len(montos_adjudicados),
    "monto_minimo": min(montos_adjudicados),
    "monto_maximo": max(montos_adjudicados),
    "monto_promedio": sum(montos_adjudicados) / len(montos_adjudicados),
    "monto_total": sum(montos_adjudicados)
}

{'cantidad_montos': 78,
 'monto_minimo': 7430.71,
 'monto_maximo': 474000,
 'monto_promedio': 80329.25833333333,
 'monto_total': 6265682.15}